In [1]:
import requests
import time
import pandas as pd

In [2]:
df = pd.read_csv('./Resources/NY-House-Dataset.csv')

In [3]:
#display the first 5 rows of the dataframe
df.head()

,BROKERTITLE,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,MAIN_ADDRESS,ADMINISTRATIVE_AREA_LEVEL_2,LOCALITY,SUBLOCALITY,STREET_NAME,LONG_NAME,FORMATTED_ADDRESS,LATITUDE,LONGITUDE
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.0,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.0,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.0,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.0,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.0,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856


In [4]:

# Your API key
api_key = 'REDACTED_GEOAPIFY_KEY'  

# Initialize a list to store the results
results = []

In [6]:
test_lat = 40.748817  # Latitude for the Empire State Building
test_lon = -73.985428  # Longitude for the Empire State Building

url = f'https://api.geoapify.com/v1/geocode/reverse?lat={test_lat}&lon={test_lon}&format=json&apiKey={api_key}'

response = requests.get(url)
print(response.json())

{'results': [{'country_code': 'us', 'name': 'West 34th Street & 5th Avenue', 'country': 'United States', 'county': 'New York County', 'datasource': {'sourcename': 'openstreetmap', 'attribution': '© OpenStreetMap contributors', 'license': 'Open Database License', 'url': 'https://www.openstreetmap.org/copyright'}, 'state': 'New York', 'state_code': 'NY', 'city': 'New York', 'district': 'Manhattan', 'suburb': 'Midtown West', 'lon': -73.985401, 'lat': 40.74889, 'distance': 8.429871270670823, 'result_type': 'amenity', 'housenumber': '7', 'street': 'West 34th Street', 'postcode': '10001', 'formatted': 'West 34th Street & 5th Avenue, 7 West 34th Street, New York, NY 10001, United States of America', 'address_line1': 'West 34th Street & 5th Avenue', 'address_line2': '7 West 34th Street, New York, NY 10001, United States of America', 'timezone': {'name': 'America/New_York', 'offset_STD': '-05:00', 'offset_STD_seconds': -18000, 'offset_DST': '-04:00', 'offset_DST_seconds': -14400, 'abbreviation_

In [5]:

# Loop through each row in the DataFrame
for index, row in df.iterrows():
    lat = row['LATITUDE']
    lon = row['LONGITUDE']

    # Construct the API URL
    url = f'https://api.geoapify.com/v1/geocode/reverse?lat={lat}&lon={lon}&format=json&apiKey={api_key}'

    try:
        # Make the API call
        response = requests.get(url)
        response.raise_for_status()  # Raise an HTTPError for bad responses

        data = response.json()

        if data.get('results'):
            # Extract relevant fields from the first result
            feature = data['results'][0]
            extracted_data = {
                'LATITUDE': lat,
                'LONGITUDE': lon,
                'STATE': feature.get('state', None),
                'LOCALITY': feature.get('city', feature.get('town', feature.get('village', None))),
                'SUBLOCALITY': feature.get('suburb', None),
                'POSTCODE': feature.get('postcode', None),
                'PROPERTY_CATEGORY': feature.get('result_type', None),
                'GEOCODING_TYPE': feature.get('result_type', None)
            }
            results.append(extracted_data)
        else:
            print(f"No results for lat: {lat}, lon: {lon}")
            results.append({
                'LATITUDE': lat,
                'LONGITUDE': lon,
                'STATE': None,
                'LOCALITY': None,
                'SUBLOCALITY': None,
                'POSTCODE': None,
                'PROPERTY_CATEGORY': None,
                'GEOCODING_TYPE': None
            })

    except requests.exceptions.RequestException as e:
        print(f"Error for lat: {lat}, lon: {lon} - {e}")
        results.append({
            'LATITUDE': lat,
            'LONGITUDE': lon,
            'STATE': None,
            'LOCALITY': None,
            'SUBLOCALITY': None,
            'POSTCODE': None,
            'PROPERTY_CATEGORY': None,
            'GEOCODING_TYPE': None
        })

    # Sleep to avoid hitting API rate limits
    time.sleep(1)  # Adjust based on API limits

# Convert the results into a DataFrame
api_results_df = pd.DataFrame(results)

# Display the first few rows
print(api_results_df.head())


Error for lat: 40.8426571, lon: -73.8884167 - 524 Server Error:  for url: https://api.geoapify.com/v1/geocode/reverse?lat=40.8426571&lon=-73.8884167&format=json&apiKey=REDACTED_GEOAPIFY_KEY
    LATITUDE  LONGITUDE     STATE  LOCALITY   SUBLOCALITY POSTCODE  \
0  40.761255 -73.974483  New York  New York  Midtown East    10022   
1  40.766393 -73.980991  New York  New York          None    10019   
2  40.541805 -74.196109  New York  New York          None    10312   
3  40.761398 -73.974613  New York  New York  Midtown East    10022   
4  40.767224 -73.969856  New York  New York          None    10065   

  PROPERTY_CATEGORY GEOCODING_TYPE  
0           amenity        amenity  
1           amenity        amenity  
2          building       building  
3           amenity        amenity  
4          building       building  


In [1]:
# Convert the results to a DataFrame
api_results_df = pd.DataFrame(results)
api_results_df.head()


NameError: name 'pd' is not defined

In [13]:
#save the extarcted data to a csv file
api_results_df.to_csv('./Resources/housing_geocode_extraction.csv', index=False)  # Save without the index